In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm
from google.colab import files

# Install kagglehub if not already present in the Colab environment
try:
    import kagglehub
except ImportError:
    import subprocess
    subprocess.run(["pip", "install", "kagglehub"], check=True)
    import kagglehub

# ==========================================
# 1. KAGGLEHUB DATA DOWNLOAD
# ==========================================
print("=== STEP 1: DOWNLOADING DATASET ===")
# This automatically handles the download, extraction, and caching
path = kagglehub.dataset_download("lianesantos/chest-xray-pneumonia")
print(f"Dataset downloaded to cache: {path}")

# Dynamically locate the 'train' and 'test' directories
# (Handles cases where the dataset is nested inside a 'chest_xray' subfolder)
if "chest_xray" in os.listdir(path):
    base_data_path = os.path.join(path, "chest_xray")
else:
    base_data_path = path

train_dir = os.path.join(base_data_path, "train")
test_dir = os.path.join(base_data_path, "test")

print(f"Train directory set to: {train_dir}")
print(f"Test directory set to: {test_dir}\n")

# ==========================================
# 2. MODEL & DATASET DEFINITIONS (Matches VS Code)
# ==========================================
class MedicalImageDataset(Dataset):
    def __init__(self, data_dir: str, transform=None):
        self.data_dir = Path(data_dir)
        self.transform = transform
        self.image_paths = []
        self.labels = []
        
        self.classes = sorted([d.name for d in self.data_dir.iterdir() if d.is_dir()])
        self.class_to_idx = {cls_name: i for i, cls_name in enumerate(self.classes)}
        
        for class_name in self.classes:
            class_dir = self.data_dir / class_name
            for img_path in list(class_dir.glob("*.jpeg")) + list(class_dir.glob("*.jpg")):
                self.image_paths.append(img_path)
                self.labels.append(self.class_to_idx[class_name])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert("RGB")
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)


class ResNetTransfer(nn.Module):
    def __init__(self, num_classes: int, freeze_features: bool = True):
        super(ResNetTransfer, self).__init__()
        self.model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        
        if freeze_features:
            for param in self.model.parameters():
                param.requires_grad = False
                
        num_ftrs = self.model.fc.in_features
        self.model.fc = nn.Linear(num_ftrs, num_classes)

    def forward(self, x):
        return self.model(x)

# ==========================================
# 3. TRAINING LOOP
# ==========================================
def train_model():
    print("=== STEP 2: MODEL TRAINING ===")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Transforms
    train_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(10),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])
    test_transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    # Data Loaders using the dynamic paths from kagglehub
    train_dataset = MedicalImageDataset(train_dir, transform=train_transform)
    test_dataset = MedicalImageDataset(test_dir, transform=test_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2)

    # Initialize Model
    model = ResNetTransfer(num_classes=len(train_dataset.classes)).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 3 # Increase to 5-10 for higher accuracy

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        
        for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]"):
            images, labels = images.to(device), labels.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * images.size(0)

        # Validation Phase
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for images, labels in test_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()
                
        print(f"Epoch {epoch+1} Complete - Test Accuracy: {correct/total:.4f}")

    # ==========================================
    # 4. EXPORT AND DOWNLOAD
    # ==========================================
    print("\n=== STEP 3: EXPORTING WEIGHTS ===")
    save_path = "model.pth"
    torch.save(model.state_dict(), save_path)
    print(f"Weights successfully saved to {save_path}. Initiating download to your computer...")
    
    files.download(save_path)

if __name__ == "__main__":
    train_model()